# 02 - Limpieza de Datos

Este notebook es el **punto oficial de entrada** para la fase de limpieza y análisis.

**Input principal:** `data/interim/sample_ready_cross_section.csv` (86 países, generado por `src/build_sample_ready.py`)

**Jerarquía de variables:**
- **Y principal:** ai_readiness_score, ai_adoption_rate, ai_investment_usd_bn_cumulative, ai_startups_cumulative
- **X1 principal:** has_ai_law, regulatory_approach, regulatory_intensity, enforcement_level, thematic_coverage
- **X2 core:** gdp_per_capita_ppp, internet_penetration, gii_score, oecd_member, region
- **X2 extended:** rd_expenditure, tertiary_education
- **Robustez:** ai_patents_per100k, government_effectiveness, ai_investment_vc_proxy

**Muestras disponibles:**
- PRINCIPAL (72/86): todas las Y principales + X1 + X2 core completas
- EXTENDED (62/86): PRINCIPAL + R&D + educación terciaria
- STRICT (46/86): todas las variables incluyendo robustez

**Documentación:** ver `info_data/ETL_RUNBOOK.md`, `info_data/DATA_DECISIONS_LOG.md`, `info_data/GUIA_VARIABLES_ESTUDIO_ETL.md`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
INTERIM_PATH = PROJECT_ROOT / "data" / "interim"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

# ── Cargar dataset sample-ready ──────────────────────────────────────
df = pd.read_csv(INTERIM_PATH / "sample_ready_cross_section.csv")
print(f"Dataset cargado: {df.shape[0]} países × {df.shape[1]} columnas")
print(f"\nMuestras disponibles:")
print(f"  PRINCIPAL (Y+X1+X2 core completas): {df['complete_principal'].sum()}/86")
print(f"  EXTENDED  (+R&D +education):         {df['complete_extended'].sum()}/86")
print(f"  STRICT    (todas las variables):     {df['complete_strict'].sum()}/86")
print(f"\nRepresentación regulatoria (muestra PRINCIPAL):")
principal = df[df['complete_principal'] == True]
print(principal['regulatory_status_group'].value_counts().to_string())

## 2.1 Diagnóstico de cobertura y tipos de datos

In [ ]:
# ── Variables del modelo principal ────────────────────────────────────
VARS_Y = ['ai_readiness_score', 'ai_adoption_rate', 'ai_investment_usd_bn_cumulative', 'ai_startups_cumulative']
VARS_X1 = ['has_ai_law', 'regulatory_approach', 'regulatory_intensity', 'enforcement_level', 'thematic_coverage']
VARS_X2_CORE = ['gdp_per_capita_ppp', 'internet_penetration', 'gii_score', 'oecd_member', 'region']
VARS_X2_EXT = ['rd_expenditure', 'tertiary_education']
VARS_ROBUST = ['ai_patents_per100k', 'government_effectiveness', 'ai_investment_vc_proxy']

ALL_MODEL_VARS = VARS_Y + VARS_X1 + VARS_X2_CORE + VARS_X2_EXT + VARS_ROBUST

# ── Cobertura por variable ───────────────────────────────────────────
coverage = df[ALL_MODEL_VARS].notna().sum().sort_values(ascending=False)
print("Cobertura por variable (de 86 países):")
print(coverage.to_string())

# ── Tipos de datos ───────────────────────────────────────────────────
print(f"\nTipos de datos:")
print(df[ALL_MODEL_VARS].dtypes.to_string())

# ── Valores únicos en categóricas ───────────────────────────────────
for col in ['regulatory_approach', 'regulatory_status_group', 'enforcement_level', 'region']:
    if col in df.columns:
        print(f"\n{col}: {dict(df[col].value_counts())}")

## 2.2 Subconjuntos de trabajo

Definir las muestras de trabajo según el nivel de completitud. La muestra **PRINCIPAL** es la base para el modelo central; la muestra **EXTENDED** agrega controles de capital humano; la **STRICT** incluye robustez.

In [ ]:
# ── Subconjuntos de trabajo ───────────────────────────────────────────
df_principal = df[df['complete_principal'] == True].copy()
df_extended = df[df['complete_extended'] == True].copy()
df_strict = df[df['complete_strict'] == True].copy()

print(f"df_principal: {len(df_principal)} países")
print(f"df_extended:  {len(df_extended)} países")
print(f"df_strict:    {len(df_strict)} países")

# ── Verificar representación regulatoria ─────────────────────────────
print(f"\nRepresentación regulatoria por muestra:")
for name, subset in [('PRINCIPAL', df_principal), ('EXTENDED', df_extended), ('STRICT', df_strict)]:
    counts = subset['regulatory_status_group'].value_counts()
    print(f"\n  {name} (N={len(subset)}):")
    for group, n in counts.items():
        print(f"    {group}: {n}")